# Case study parsing — run everything here (Colab GPU)

**If your birth-certificate images live in this repo** (e.g. `data/raw_images/DataSet/Birth Certificate/`), you **do not need Google Drive**. Push the repo to GitHub **including `data/`**, then use **`SOURCE = "github"`** — Colab will `git clone` and the images come with it.

**Pick one:**

- **`github`** (recommended when images are in git) — push your repo, set `REPO_URL`, clone on Colab.
- **`drive`** — only if the full project folder exists only on Google Drive, not on GitHub.

**Colab:** [colab.research.google.com](https://colab.research.google.com) → upload **this notebook** → **Runtime → Change runtime type → GPU** → run top to bottom.

**GitHub limits:** any single file **> 100 MB** cannot be pushed unless you use **Git LFS**. Huge folders may need Drive + optional copy step instead.

In [1]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

torch: 2.4.1+cpu
cuda available: False


## 1) Configure — edit this cell

**Images inside the repo:** keep **`SOURCE = "github"`**, then:

1. Push this project to GitHub (including `data/raw_images/...`).
2. On GitHub open your repo → green **Code** → **HTTPS** → copy the URL (looks like `https://github.com/you/repo.git`).
3. In the next cell set **`REPO_URL = "https://github.com/..."`** (same string, in quotes).

**Only use `SOURCE = "drive"`** if you never push to GitHub and keep the project copy only on Google Drive.

In [9]:
# --- edit below ---
import os
import tempfile
from pathlib import Path

SOURCE = "github"  # "github" = clone repo (images in repo) | "drive" = project only on Google Drive

# If SOURCE == "github": REQUIRED — paste your repo URL from GitHub → Code → HTTPS (must not stay empty).
REPO_URL = "https://github.com/iMalakAhmed/Case-Study-Parsing-.git"
# Colab: clone under /content. Local Windows/Mac: use temp (avoid "/content" → invalid path)
if os.path.isdir("/content"):
    CLONE_DIR = "/content/Case-Study-Parsing-"
else:
    CLONE_DIR = str(Path(tempfile.gettempdir()) / "Case-Study-Parsing-clone")

# If SOURCE == "drive": path to the repo root (must contain case_study_parser/, data/, ...)
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/GRAD/AI/Case-Study-Parsing-"

# If you cloned from GitHub but BC images only exist on Drive, set these after mounting Drive:
COPY_BC_FROM_DRIVE = False
DRIVE_BC_FOLDER = "/content/drive/MyDrive/path/to/Birth Certificate"
REPO_BC_DEST = "data/raw_images/DataSet/Birth Certificate"

# Section 6 — vision-language model (Hugging Face id). 3B is faster/smaller download than 7B for tests.
VL_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"  # or "Qwen/Qwen2.5-VL-7B-Instruct" for higher quality

# Section 6: use the two-image manifest (data/manifest_birth_certificate_two.json) or scan the BC folder.
BC_EXTRACTION_SOURCE = "two_bc_manifest"  # "two_bc_manifest" | "typed_dir"

# Section 6 only when BC_EXTRACTION_SOURCE == "typed_dir": cap how many BC images (None = all).
MAX_BC_DOCUMENTS = 2  # set None for every image; use 1–3 for a quick smoke test
# --- stop editing ---

## 2) Open the project (clone **or** Drive)

Clones from GitHub **or** opens your folder on Drive. Birth-certificate images are **found automatically** in step 6 (canonical paths + `BC_*.jpeg` anywhere under the repo).

In [10]:
import os
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.error
import urllib.request
import zipfile


def parse_github_owner_repo(https_url: str):
    u = https_url.strip().rstrip("/")
    if u.endswith(".git"):
        u = u[:-4]
    m = re.match(r"https?://github\.com/([^/]+)/([^/]+)/?$", u)
    if not m:
        return None
    return m.group(1), m.group(2)


def clone_or_download_github(url: str, dest: str) -> None:
    # If the kernel's cwd was inside `dest`, deleting dest breaks git subprocess:
    # fatal: Unable to read current working directory
    # Colab: /content — local Windows/Mac: system temp (never hardcode /tmp)
    safe_base = "/content" if os.path.isdir("/content") else tempfile.gettempdir()
    os.chdir(safe_base)

    if os.path.lexists(dest):
        shutil.rmtree(dest, ignore_errors=True)

    stderr_all: list[str] = []
    for cmd in (
        ["git", "clone", "--depth", "1", url, dest],
        ["git", "clone", url, dest],
    ):
        print("$", " ".join(cmd))
        proc = subprocess.run(cmd, capture_output=True, text=True, cwd=safe_base)
        if proc.stdout:
            print(proc.stdout, end="")
        if proc.stderr:
            print(proc.stderr, end="", file=sys.stderr)
            stderr_all.append(f"{' '.join(cmd)} stderr:\n{proc.stderr}")
        if proc.returncode == 0:
            return

    parsed = parse_github_owner_repo(url)
    if parsed:
        owner, repo = parsed
        print(f"Trying ZIP download (public repos only): {owner}/{repo} …")
        zip_path = os.path.join(tempfile.gettempdir(), "github_repo_dl.zip")
        unpack_root = os.path.join(tempfile.gettempdir(), "github_zip_root")
        shutil.rmtree(unpack_root, ignore_errors=True)
        for branch in ("main", "master"):
            zip_url = f"https://github.com/{owner}/{repo}/archive/refs/heads/{branch}.zip"
            try:
                urllib.request.urlretrieve(zip_url, zip_path)
                if os.path.getsize(zip_path) < 500:
                    stderr_all.append(f"ZIP {branch}: tiny file (wrong branch?)")
                    continue
                with zipfile.ZipFile(zip_path, "r") as zf:
                    top = zf.namelist()[0].split("/")[0]
                    zf.extractall(unpack_root)
                inner = os.path.join(unpack_root, top)
                shutil.move(inner, dest)
                print("ZIP fallback OK.")
                return
            except (urllib.error.HTTPError, OSError, zipfile.BadZipFile) as exc:
                stderr_all.append(f"ZIP {branch}: {exc}")
                continue
            finally:
                shutil.rmtree(unpack_root, ignore_errors=True)

    raise RuntimeError(
        "Could not get the repo. Full diagnostics:\n\n"
        + "\n\n".join(stderr_all)
        + f"\n\nFixes: (1) Private repo: REPO_URL = https://TOKEN@github.com/owner/repo.git "
        f"(2) Run !rm -rf {dest!r} then retry (3) Confirm URL in browser."
    )


if SOURCE == "github":
    url = REPO_URL.strip()
    if not url.startswith(("https://", "http://")):
        raise ValueError(
            "REPO_URL is empty or not an http(s) URL. On GitHub: Code → HTTPS → copy → paste into "
            'REPO_URL in the cell above. Push your repo first. Or set SOURCE = "drive" if you only use Google Drive.'
        )
    clone_or_download_github(url, CLONE_DIR)
    os.chdir(CLONE_DIR)
elif SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT_PATH):
        raise FileNotFoundError(DRIVE_PROJECT_PATH)
    os.chdir(DRIVE_PROJECT_PATH)
else:
    raise ValueError('SOURCE must be "github" or "drive"')

print("cwd:", os.getcwd())


$ git clone --depth 1 https://github.com/iMalakAhmed/Case-Study-Parsing-.git C:\Users\ASUS\AppData\Local\Temp\Case-Study-Parsing-clone


fatal: destination path 'C:\Users\ASUS\AppData\Local\Temp\Case-Study-Parsing-clone' already exists and is not an empty directory.
fatal: destination path 'C:\Users\ASUS\AppData\Local\Temp\Case-Study-Parsing-clone' already exists and is not an empty directory.


$ git clone https://github.com/iMalakAhmed/Case-Study-Parsing-.git C:\Users\ASUS\AppData\Local\Temp\Case-Study-Parsing-clone
Trying ZIP download (public repos only): iMalakAhmed/Case-Study-Parsing- …
ZIP fallback OK.
cwd: C:\Users\ASUS\AppData\Local\Temp\Case-Study-Parsing-clone


## 3) Optional — copy birth certificates from Drive (GitHub clone only)

Skip if `data/raw_images/DataSet/Birth Certificate/` is already in the repo.

Set **`COPY_BC_FROM_DRIVE = True`** in the config cell and fix **`DRIVE_BC_FOLDER`**. This cell mounts Drive if needed.

In [11]:
import os
import shutil

if not COPY_BC_FROM_DRIVE:
    print("Skipping (COPY_BC_FROM_DRIVE is False).")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = os.path.join(os.getcwd(), REPO_BC_DEST)
    os.makedirs(dest, exist_ok=True)
    for name in os.listdir(DRIVE_BC_FOLDER):
        src = os.path.join(DRIVE_BC_FOLDER, name)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(dest, name))
    print("Copied BC images to:", dest)

Skipping (COPY_BC_FROM_DRIVE is False).


## 4) Install dependencies

Run **section 2** first. This cell looks for `requirements.txt` in the clone, **downloads it from GitHub raw** if missing, or installs a **built-in fallback** list (same as your local `requirements.txt`) — then **`git push requirements.txt`** when you can.

In [12]:
import os
import re
import subprocess
import sys
import tempfile
import urllib.request
from pathlib import Path


def find_requirements_file() -> tuple[str, str]:
    """Return (directory_to_chdir, path_to_requirements_txt)."""
    cwd_req = Path(os.getcwd()) / "requirements.txt"
    if cwd_req.is_file():
        return str(os.getcwd()), str(cwd_req)

    bases = []
    if SOURCE == "drive":
        bases.append(Path(DRIVE_PROJECT_PATH))
    bases.append(Path(CLONE_DIR))

    for base in bases:
        if not base.is_dir():
            continue
        direct = base / "requirements.txt"
        if direct.is_file():
            return str(base), str(direct)
        nested = next(base.rglob("requirements.txt"), None)
        if nested is not None:
            return str(nested.parent), str(nested)

    u = REPO_URL.strip().rstrip("/")
    if u.endswith(".git"):
        u = u[:-4]
    m = re.match(r"https?://github\.com/([^/]+)/([^/]+)/?$", u)
    if m:
        owner, repo = m.group(1), m.group(2)
        for branch in ("main", "master"):
            raw = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/requirements.txt"
            tmp = os.path.join(tempfile.gettempdir(), "requirements_from_github.txt")
            try:
                urllib.request.urlretrieve(raw, tmp)
                if os.path.getsize(tmp) > 80:
                    if Path(CLONE_DIR).is_dir():
                        work = CLONE_DIR
                    else:
                        work = "/content" if os.path.isdir("/content") else tempfile.gettempdir()
                    return work, tmp
            except OSError:
                continue

    return "", ""


root, req_path = find_requirements_file()
if not req_path:
    pinned = """accelerate>=0.33.0
jsonschema>=4.23.0
Pillow>=10.0.0
torch>=2.4.0
transformers>=4.51.0
"""
    req_path = os.path.join(tempfile.gettempdir(), "requirements_fallback.txt")
    Path(req_path).write_text(pinned, encoding="utf-8")
    root = CLONE_DIR if Path(CLONE_DIR).is_dir() else os.getcwd()
    print(
        "NOTE: requirements.txt not in clone/GitHub — installing built-in fallback deps. "
        "Push requirements.txt from your PC: git add requirements.txt && git push"
    )

if root and Path(root).is_dir():
    os.chdir(root)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", req_path],
    check=True,
)


CompletedProcess(args=['c:\\Users\\ASUS\\AppData\\Local\\Programs\\Python\\Python311\\python.exe', '-m', 'pip', 'install', '-q', '-r', 'C:\\Users\\ASUS\\AppData\\Local\\Temp\\Case-Study-Parsing-clone\\Case-Study-Parsing--main\\requirements.txt'], returncode=0)

## 5) Optional — Hugging Face login

Uncomment if the model requires authentication.

In [ ]:
# from huggingface_hub import login
# login()

## 6) Run birth-certificate extraction

Writes **`outputs/predictions/`** and **`outputs/raw_model_outputs/`**.

**Output shape:** For **`document_type: "birth_certificate"`**, each prediction JSON has three top-level groups — **`birth_certificate`** (registration/office metadata), **`ids`** (national ID numbers), **`personal_and_other`** (names, gender, religion, nationality, birth place/date, misc). See **`schemas/birth_certificate.schema.json`** and **`data/birth_certificate_bundle.json`** (`birth_certificate_extraction_template`).

**Manifest:** With **`BC_EXTRACTION_SOURCE = "two_bc_manifest"`**, runs **`data/birth_certificate_bundle.json`** (`pipeline_manifest`). Validate outputs: **`python -m case_study_parser.validate --input-dir outputs/predictions --birth-certificate`**.

**Same kernel, no reload:** **`run_extract`** is in-process; re-run this cell to re-infer without reloading the model. **`clear_model_cache()`** frees GPU RAM.

**Time:** First run downloads weights. Check **`[extract] cuda_available=True`**.

If you see **cannot find case_study_parser**, push the package. If **images** are missing, push **`data/`** and re-run from step 2.

In [13]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path


def refresh_stale_github_clone() -> None:
    """Sync CLONE_DIR when package is missing or common.py lacks birth-certificate helpers."""
    if SOURCE != "github":
        return
    clone = Path(CLONE_DIR)
    if not (clone / ".git").is_dir():
        return

    def _common_paths() -> list[Path]:
        try:
            return [p for p in clone.rglob("case_study_parser/common.py") if p.is_file()]
        except OSError:
            return []

    def _has_bc_helpers() -> bool:
        for p in _common_paths():
            try:
                if "def apply_birth_certificate_defaults" in p.read_text(encoding="utf-8", errors="replace"):
                    return True
            except OSError:
                continue
        return False

    def _has_package() -> bool:
        try:
            return any(p.is_file() for p in clone.rglob("case_study_parser/__init__.py"))
        except OSError:
            return False

    if _has_package() and _has_bc_helpers():
        return

    safe_base = "/content" if Path("/content").is_dir() else tempfile.gettempdir()
    try:
        os.chdir(safe_base)
    except OSError:
        pass
    for branch in ("main", "master"):
        fe = subprocess.run(
            ["git", "-C", str(clone), "fetch", "--depth", "1", "origin", branch],
            capture_output=True,
            text=True,
        )
        if fe.returncode != 0:
            continue
        subprocess.run(
            ["git", "-C", str(clone), "reset", "--hard", f"origin/{branch}"],
            capture_output=True,
            text=True,
        )
        if _has_package() and _has_bc_helpers():
            print(f"Synced clone to origin/{branch} (birth-certificate helpers present).")
            return


def _repo_root_if_packaged(base: Path) -> Path | None:
    """Return dir if ``base`` or one subdirectory contains ``case_study_parser/__init__.py``."""
    try:
        base = base.expanduser().resolve()
    except OSError:
        return None
    if (base / "case_study_parser" / "__init__.py").is_file():
        return base
    try:
        for child in sorted(base.iterdir()):
            if child.is_dir() and not child.name.startswith("."):
                if (child / "case_study_parser" / "__init__.py").is_file():
                    return child
    except OSError:
        pass
    return None


def resolve_repo_root() -> Path:
    """Find repo root (directory containing case_study_parser/)."""
    # Local Jupyter: cwd may be notebooks/ — walk parents until we find the package
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        pkg = base / "case_study_parser"
        if pkg.is_dir() and (pkg / "__init__.py").is_file():
            os.chdir(base)
            return base.resolve()

    candidates = [
        Path(CLONE_DIR),
        Path(DRIVE_PROJECT_PATH) if SOURCE == "drive" else None,
        Path("/content/Case-Study-Parsing-"),
    ]
    for base in candidates:
        if base is None:
            continue
        found = _repo_root_if_packaged(Path(base))
        if found is not None:
            os.chdir(found)
            return found.resolve()

    # GitHub-ZIP style tree: CLONE_DIR/<repo>-main/case_study_parser/…
    cp = Path(CLONE_DIR)
    if cp.is_dir():
        try:
            matches = [p.parent.parent for p in cp.rglob("case_study_parser/__init__.py") if p.is_file()]
            if matches:
                best = min(matches, key=lambda p: len(p.parts))
                os.chdir(best)
                return best.resolve()
        except OSError:
            pass

    content_root = Path("/content")
    if content_root.is_dir():
        try:
            for marker in content_root.rglob("case_study_parser/__init__.py"):
                base = marker.parent.parent
                os.chdir(base)
                return base.resolve()
        except OSError:
            pass

    hint_lines = [
        f"No case_study_parser/ walking upward from cwd: {here}",
    ]
    cp = Path(CLONE_DIR)
    if cp.is_dir():
        try:
            names = sorted(p.name for p in cp.iterdir())
            hint_lines.append(f"Contents of {cp}: {names}")
        except OSError as e:
            hint_lines.append(f"Could not list {cp}: {e}")
    else:
        hint_lines.append(
            f"Clone path does not exist: {cp} — run section 2, or open this notebook from inside the repo."
        )

    raise FileNotFoundError(
        "Cannot find case_study_parser/ (with __init__.py). "
        "Either push the package from your PC, or re-run section 2 to re-clone (stale Colab folder).\n\n"
        "On your PC, from the project folder, run:\n"
        "  git add case_study_parser prompts schemas data requirements.txt\n"
        "  git commit -m \"Add pipeline code and data\"\n"
        "  git push\n\n"
        + ("\n".join(hint_lines))
    )


refresh_stale_github_clone()
repo_root = resolve_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

_bc_common = repo_root / "case_study_parser" / "common.py"
if _bc_common.is_file():
    _bc_txt = _bc_common.read_text(encoding="utf-8", errors="replace")
    if "def apply_birth_certificate_defaults" not in _bc_txt:
        raise ImportError(
            "This checkout is missing birth-certificate helpers in case_study_parser/common.py "
            "(extract.py is newer than common.py — stale or partial clone). "
            "Fix: (1) Delete the clone folder (e.g. Temp\\\\Case-Study-Parsing-clone), "
            "re-run section 2, then section 6. "
            "(2) Or from your PC: git pull, commit, push so GitHub has one consistent tree. "
            f"repo_root={repo_root!s}"
        )

from case_study_parser.common import find_birth_certificate_image_dir
from case_study_parser.extract import build_parser, run_extract

manifest_path = repo_root / "data" / "birth_certificate_bundle.json"

if BC_EXTRACTION_SOURCE == "two_bc_manifest":
    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"Missing {manifest_path}. Pull latest repo or add data/birth_certificate_bundle.json."
        )
    print("BC_EXTRACTION_SOURCE=two_bc_manifest ->", manifest_path.relative_to(repo_root))
else:
    root = Path.cwd()
    bc_dir = find_birth_certificate_image_dir(root)
    if bc_dir is None:
        top = list(root.iterdir()) if root.is_dir() else []
        raise FileNotFoundError(
            "Could not find birth certificate images (searched for "
            "data/raw_images/DataSet/Birth Certificate, BC_*.jpeg under repo, "
            "and folders named Birth Certificate).\n"
            f"Repo root has: {[p.name for p in top]}\n"
            "Fix: git add data && git push"
        )
    rel = bc_dir.relative_to(root).as_posix()
    print("Using birth_certificate=", rel)
    if MAX_BC_DOCUMENTS is not None:
        print(f"MAX_BC_DOCUMENTS={MAX_BC_DOCUMENTS} (set None in config for full dataset)")
    else:
        print("MAX_BC_DOCUMENTS=None — processing every image (may take a long time)")

try:
    import torch as _torch

    print(
        "Notebook kernel:",
        "cuda=",
        _torch.cuda.is_available(),
        "|",
        _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else "no CUDA",
    )
except Exception as _e:
    print("Notebook kernel torch check:", _e)

print("VL_MODEL=", VL_MODEL)

if BC_EXTRACTION_SOURCE == "two_bc_manifest":
    argv = [
        "--manifest",
        str(manifest_path),
        "--output-dir",
        "outputs",
        "--model-name",
        VL_MODEL,
        "--typed-model",
        f"birth_certificate={VL_MODEL}",
        "--torch-dtype",
        "float16",
        "--trust-remote-code",
    ]
else:
    argv = [
        "--typed-input-dir",
        f"birth_certificate={rel}",
        "--output-dir",
        "outputs",
        "--model-name",
        VL_MODEL,
        "--typed-model",
        f"birth_certificate={VL_MODEL}",
        "--torch-dtype",
        "float16",
        "--trust-remote-code",
    ]
    if MAX_BC_DOCUMENTS is not None:
        argv.extend(["--max-documents", str(MAX_BC_DOCUMENTS)])

print(
    "\n--- extract (in-process; model stays in RAM — re-run this cell to re-infer without reload) ---\n",
    flush=True,
)
run_extract(build_parser().parse_args(argv))


c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'apply_birth_certificate_defaults' from 'case_study_parser.common' (C:\Users\ASUS\AppData\Local\Temp\Case-Study-Parsing-clone\Case-Study-Parsing--main\case_study_parser\common.py)

## 7) Zip outputs (download on Colab)

Download **`outputs.zip`** before the runtime disconnects.

In [8]:
import tempfile
import zipfile
from pathlib import Path

zip_path = Path(tempfile.gettempdir()) / "outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("outputs/predictions", "outputs/raw_model_outputs"):
        base = Path(folder)
        if base.is_dir():
            for f in base.rglob("*"):
                if f.is_file():
                    zf.write(f, f.as_posix())
print("Wrote", zip_path.resolve())
try:
    from google.colab import files

    files.download(str(zip_path))
except ImportError:
    print("Not on Colab — open outputs/ locally or:", zip_path.resolve())

Wrote C:\Users\ASUS\AppData\Local\Temp\outputs.zip
Not on Colab — open outputs/ locally or: C:\Users\ASUS\AppData\Local\Temp\outputs.zip
